# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vrindasharma634-max/flyrank1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional Lane:** Refresh / Content Opportunity Scoring

**Why this lane:**  
Content decay is an inevitable challenge in search marketing: after publication, search rankings and organic impressions naturally erode due to competitor updates, shifts in search algorithms, and evolving searcher intent. Editorial teams overseeing catalogs of thousands of URLs cannot manually inspect every piece of content every month. Simple calendar rules (such as *"refresh everything older than 180 days"*) fail because decay is non-linear—many evergreen pages maintain strong visibility indefinitely, whereas pages in competitive or volatile topics can drop sharply within weeks. This lane focuses on building an evidence-based, machine-learned opportunity scoring system that ranks declining pages by true exposure risk and recovery potential, providing content teams with a transparent, prioritized review queue.

In [1]:
import os
from pathlib import Path
import pandas as pd

# Handle both root directory execution and work/notebooks/ execution
data_path = Path("data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("=== Lane Setup & Dataset Verification ===")
print("Selected Lane: Refresh / Content Opportunity Scoring")
print(f"Dataset path:  {data_path}")
print(f"Dimensions:    {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Client count:  {df['client_id'].nunique()} distinct pseudonymized clients")

=== Lane Setup & Dataset Verification ===
Selected Lane: Refresh / Content Opportunity Scoring
Dataset path:  ../../data/raw/content_refresh_anonymized.csv
Dimensions:    30,000 rows × 44 columns
Client count:  32 distinct pseudonymized clients


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**1. The Decision:**  
Which published content item should an editorial or content marketing team review and update first to prevent or reverse organic traffic decay?

**2. Who Acts and What Action They Take:**  
The primary user is a content editor, SEO lead, or content strategist. Based on the ranked queue and associated reason codes, the team performs a targeted content refresh: updating outdated factual information and statistics, improving internal link structure, rewriting meta titles and descriptions to boost CTR, expanding thin content sections, or re-aligning search intent with current SERP requirements.

**3. Cost of a Wrong Recommendation:**  
* **False Positive (recommending a page that did not need a refresh):** Wastes scarce editorial budget and writer time (typically 2 to 6 hours per piece) rewriting content that was either stable, temporarily fluctuating due to seasonal search volume, or lacking sufficient baseline demand to yield an incremental return.
* **False Negative (failing to flag a high-value page undergoing rapid decay):** Causes sustained loss of Page 1 rankings and organic search traffic. Once a page drops off top search positions, re-establishing rank and authority requires significantly more effort and expenditure than proactive intervention.

**4. Why Plain Rules Are Not Enough:**  
Static rules (e.g., flagging any page with `days_since_last_update >= 180` or `ctr < 0.5%`) are brittle. In practice, content decay is driven by non-linear interactions between historical impression momentum, keyword search volume, average position on the SERP, and on-page user engagement. Machine learning learns these tangled patterns directly from data to prioritize candidate pages far more accurately than fixed thresholds.

**Framing Summary:**  
> For **content editors and SEO leads**, deciding **which declining pages to refresh first**, we will build a **ranked opportunity scoring queue** from **pre-decision search and analytics signals**, scoring **decline risk and refresh opportunity** measured by **Precision@50 and Average Precision against holdout clients**. A wrong call costs **wasted editorial bandwidth** (false positive) or **permanent organic visibility loss on core assets** (false negative). Plain rules are insufficient because decay is multi-factorial; we will claim strictly **observational, decision-support** recommendations.

In [2]:
# Quantifying the editorial capacity bottleneck: Why prioritization is needed
total_pages = len(df)
declining_mask = df["trend_direction"] == "down"
declining_count = int(declining_mask.sum())
high_visibility_declining = int((declining_mask & (df["impressions_90d"] >= 500)).sum())
page1_striking_declining = int((declining_mask & (df["avg_position"] > 0) & (df["avg_position"] <= 20)).sum())

print("=== The Editorial Prioritization Bottleneck ===")
print(f"Total content items in catalog:                    {total_pages:,}")
print(f"Total declining items (trend_direction == 'down'): {declining_count:,} ({declining_count / total_pages:.1%})")
print(f"Declining items with >= 500 impressions/90d:       {high_visibility_declining:,} ({high_visibility_declining / total_pages:.1%})")
print(f"Declining items in Page 1 / Striking distance:     {page1_striking_declining:,} ({page1_striking_declining / total_pages:.1%})")
print()
print("Takeaway: An editorial team cannot manually inspect 16,000+ decaying pages.")
print("Even restricting to high-demand pages yields thousands of candidates.")
print("A machine-learned opportunity rank order directly resolves this capacity constraint.")

=== The Editorial Prioritization Bottleneck ===
Total content items in catalog:                    30,000
Total declining items (trend_direction == 'down'): 16,262 (54.2%)
Declining items with >= 500 impressions/90d:       9,961 (33.2%)
Declining items in Page 1 / Striking distance:     11,744 (39.1%)

Takeaway: An editorial team cannot manually inspect 16,000+ decaying pages.
Even restricting to high-demand pages yields thousands of candidates.
A machine-learned opportunity rank order directly resolves this capacity constraint.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Analyzing the 30,000 rows in `data/raw/content_refresh_anonymized.csv` reveals three empirical facts that prove this lane addresses a substantial, high-impact problem:

1. **High prevalence of content decay:**  
   **16,262 out of 30,000 pages (54.21%)** have a `trend_direction` of `"down"` (a drop greater than 20% in impressions between the most recent 30-day window and the prior 30-day window). Decay is the dominant dynamic in this content inventory, not an isolated edge case.

2. **Substantial high-value exposure at risk:**  
   Among the declining pages, **9,961 pages (33.20% of the entire dataset)** have strong baseline demand (`impressions_90d >= 500`), and **11,744 pages (39.15% of the entire dataset)** occupy high-value search positions (Page 1 or striking distance: `0 < avg_position <= 20`). These are not abandoned zero-traffic pages; they are core revenue-driving assets currently slipping down search rankings.

3. **Massive aggregate traffic loss:**  
   Across the 16,262 declining pages, 30-day search impressions collapsed from **29,408,488** in the previous 30-day window to **15,311,594** in the most recent 30 days—a net loss of **14,096,894 impressions (-47.93%)** in a single monthly cycle. This quantified exposure confirms that intelligent refresh prioritization protects millions of impressions across clients.

In [3]:
# Calculating exact supporting metrics from data/raw/content_refresh_anonymized.csv
total_items = len(df)

# Finding 1: Base rate of decay
declining_mask = df["trend_direction"] == "down"
n_declining = int(declining_mask.sum())
pct_declining = (n_declining / total_items) * 100

# Finding 2: High-value exposure at risk
high_demand_mask = df["impressions_90d"] >= 500
striking_pos_mask = (df["avg_position"] > 0) & (df["avg_position"] <= 20)

n_declining_high_demand = int((declining_mask & high_demand_mask).sum())
pct_declining_high_demand = (n_declining_high_demand / total_items) * 100

n_declining_striking = int((declining_mask & striking_pos_mask).sum())
pct_declining_striking = (n_declining_striking / total_items) * 100

# Finding 3: Actual aggregate impression loss in 30-day comparison windows
prev_30d_impressions = int(df.loc[declining_mask, "impressions_prev_30d"].sum())
last_30d_impressions = int(df.loc[declining_mask, "impressions_last_30d"].sum())
net_impression_loss = prev_30d_impressions - last_30d_impressions
pct_volume_drop = (net_impression_loss / prev_30d_impressions) * 100

print(f"=== Real Supporting Metrics: {data_path.name} ===")
print("Finding 1 — Overall Decay Prevalence:")
print(f"  • Total Content Items: {total_items:,}")
print(f"  • Declining Items (trend_direction == 'down'): {n_declining:,} ({pct_declining:.2f}%)")
print()
print("Finding 2 — High-Exposure Items at Risk:")
print(f"  • Declining with High Demand (impressions_90d >= 500): {n_declining_high_demand:,} ({pct_declining_high_demand:.2f}%)")
print(f"  • Declining in Page 1 / Striking Position (0 < avg_pos <= 20): {n_declining_striking:,} ({pct_declining_striking:.2f}%)")
print()
print("Finding 3 — Aggregate Impression Trajectory on Declining Pages:")
print(f"  • Impressions in Previous 30-day window: {prev_30d_impressions:,}")
print(f"  • Impressions in Last 30-day window:     {last_30d_impressions:,}")
print(f"  • Net Impression Loss:                   {net_impression_loss:,} ({pct_volume_drop:.2f}% decline)")

=== Real Supporting Metrics: content_refresh_anonymized.csv ===
Finding 1 — Overall Decay Prevalence:
  • Total Content Items: 30,000
  • Declining Items (trend_direction == 'down'): 16,262 (54.21%)

Finding 2 — High-Exposure Items at Risk:
  • Declining with High Demand (impressions_90d >= 500): 9,961 (33.20%)
  • Declining in Page 1 / Striking Position (0 < avg_pos <= 20): 11,744 (39.15%)

Finding 3 — Aggregate Impression Trajectory on Declining Pages:
  • Impressions in Previous 30-day window: 29,408,488
  • Impressions in Last 30-day window:     15,311,594
  • Net Impression Loss:                   14,096,894 (47.93% decline)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

Maintaining scientific integrity requires being explicit about the boundaries of observational search data:

### What this work CAN claim:
* **Observational Correlations:** We can identify historical statistical associations between pre-decision signals (such as impression volume, position tiers, CTR relative to position, content age, and engagement) and subsequent downward momentum.
* **Decision-Support Prioritization:** We can produce a transparent, ranked queue of candidate pages ordered by estimated opportunity score and decline risk, paired with human-auditable reason codes.
* **Empirical Benchmark vs. Heuristics:** We can measure objectively whether a learned ranking model achieves higher Precision@50 and Average Precision than transparent heuristic baseline rules on held-out clients.

### What this work CANNOT claim:
* **No Causal Proof:** We cannot claim that updating or refreshing an article *caused* traffic recovery. In observational data without randomized controlled trials (A/B testing) or quasi-experimental interventions, post-update performance may be confounded by external market shifts, seasonality, or platform dynamics.
* **No Claim of "Predicting Google's Algorithm":** We do not reverse-engineer Google's proprietary search ranking algorithms. Changes in impressions and rankings are subject to external macro-factors beyond on-page quality, including search engine layout changes (e.g., AI Overviews, featured snippets), competitor activity, and macroeconomic seasonality.
* **Proxy Target Discipline:** In this starter slice, `is_declining_label` is derived from trailing 30-day comparisons (`trend_direction == "down"`). It is an observational proxy, not an intervention outcome. Furthermore, `trend_pct` and `trend_direction` must never be used as predictive features to prevent target leakage.

In [4]:
# Verifying that simple causal heuristics fail and demonstrating observational nuance
print("=== Evidence Supporting Observational Nuance ===")

# Check: Does calendar age or staleness alone explain decline?
stale_cutoff = 180
stale_pages = df[df["days_since_last_update"] >= stale_cutoff]
stale_down = int((stale_pages["trend_direction"] == "down").sum())
stale_up = int((stale_pages["trend_direction"] == "up").sum())
stale_stable = int((stale_pages["trend_direction"] == "stable").sum())

print(f"Pages not updated for >= {stale_cutoff} days: {len(stale_pages):,}")
print(f"  • Declining: {stale_down:,} ({stale_down / len(stale_pages):.1%})")
print(f"  • Trending UP: {stale_up:,} ({stale_up / len(stale_pages):.1%})")
print(f"  • Stable/Other: {stale_stable:,} ({stale_stable / len(stale_pages):.1%})")

# Confounding demonstration: 
# Many stale pages continue to gain traffic, while recently updated pages can still decline.
fresh_pages = df[df["days_since_last_update"] <= 30]
fresh_down = int((fresh_pages["trend_direction"] == "down").sum())
print(f"\nPages updated within last 30 days: {len(fresh_pages):,}")
print(f"  • Still Declining: {fresh_down:,} ({fresh_down / len(fresh_pages):.1%})")

print("\nObservational Takeaway:")
print("Content staleness does not linearly dictate performance.")
print("This confirms that decay is multi-factorial, and claims must remain strictly observational and decision-support.")

=== Evidence Supporting Observational Nuance ===
Pages not updated for >= 180 days: 174
  • Declining: 82 (47.1%)
  • Trending UP: 27 (15.5%)
  • Stable/Other: 24 (13.8%)

Pages updated within last 30 days: 20,480
  • Still Declining: 10,473 (51.1%)

Observational Takeaway:
Content staleness does not linearly dictate performance.
This confirms that decay is multi-factorial, and claims must remain strictly observational and decision-support.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.